## 🎯 Learning Objectives
* Understand the core concepts and necessity of hybrid retrieval in RAG systems.
* Implement sparse retrieval using a modern library like BM25.
* Implement dense retrieval using state-of-the-art embedding models (e.g., Sentence Transformers).
* Develop a hybrid retrieval strategy, such as Reciprocal Rank Fusion (RRF), to combine sparse and dense results.
* Benchmark the performance of sparse, dense, and hybrid retrieval methods using relevant metrics like Recall@k.
* Analyze the trade-offs and benefits of different retrieval strategies for various query types.


## Lesson RAG01-L14: Exercise - Implement and Benchmark Hybrid Retrieval

### Task Description

In this exercise, you will implement and benchmark a hybrid retrieval system. Hybrid retrieval combines the strengths of both sparse (keyword-based) and dense (vector-based) retrieval methods to improve the overall relevance and recall of retrieved documents. You will use a small, mock dataset to demonstrate the implementation and evaluate its performance against individual sparse and dense methods.

### Requirements

1.  **Mock Dataset**: Utilize the provided mock dataset of documents, queries, and ground truth mappings.
2.  **Sparse Retriever Implementation**: Implement a function for sparse retrieval using `rank_bm25`.
3.  **Dense Retriever Implementation**: Implement a function for dense retrieval using a `SentenceTransformer` model (e.g., `all-MiniLM-L6-v2`).
4.  **Hybrid Retriever Implementation**: Implement a function that combines the results from both sparse and dense retrievers. A common and effective fusion strategy is **Reciprocal Rank Fusion (RRF)**. You will need to implement RRF to merge the ranked lists.
5.  **Benchmarking**: Evaluate all three retrieval methods (sparse, dense, hybrid) using the provided `calculate_recall_at_k` function.
6.  **Analysis**: Present the benchmarking results clearly and briefly discuss your observations regarding the performance of each method.

### Evaluation Criteria

*   **Correctness**: The sparse, dense, and hybrid retrieval functions must be correctly implemented and produce reasonable results.
*   **Clarity**: Code should be well-structured, readable, and include comments where necessary.
*   **Functionality**: The benchmarking process should run without errors and produce meaningful performance metrics.
*   **Understanding**: The analysis of results should demonstrate an understanding of why hybrid retrieval can be beneficial.
*   **Modern Practices**: Adherence to modern Python and RAG library usage.


In [ ]:
# Install necessary libraries if you haven't already
# !pip install rank_bm25 sentence-transformers scikit-learn numpy

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import collections

# --- Mock Dataset Setup ---

documents = [
    "The capital of France is Paris, known for its Eiffel Tower and Louvre Museum.",
    "Artificial intelligence is a rapidly advancing field, encompassing machine learning and deep learning.",
    "Python is a versatile programming language, widely used in data science, web development, and AI.",
    "The Louvre Museum in Paris houses thousands of works of art, including the Mona Lisa.",
    "Machine learning algorithms enable systems to learn from data without explicit programming.",
    "Data structures like lists, dictionaries, and sets are fundamental concepts in Python programming.",
    "Generative AI models, like large language models, are transforming content creation and automation.",
    "The Seine River flows through Paris, adding to its scenic beauty and historical significance."
]

queries = [
    "What is the capital of France?",
    "Explain artificial intelligence concepts.",
    "Python programming fundamentals.",
    "Famous art museums in Paris.",
    "Definition of machine learning.",
    "Generative AI and large language models."
]

# Ground truth: For each query, a list of indices of relevant documents.
# This is simplified for a small dataset.
ground_truth = {
    0: [0, 3, 7],  # "What is the capital of France?" -> Paris, Louvre, Seine
    1: [1, 4, 6],  # "Explain artificial intelligence concepts." -> AI, ML, Generative AI
    2: [2, 5],     # "Python programming fundamentals." -> Python, Data structures
    3: [0, 3, 7],  # "Famous art museums in Paris." -> Paris, Louvre, Seine
    4: [1, 4],     # "Definition of machine learning." -> AI, ML
    5: [1, 6]      # "Generative AI and large language models." -> AI, Generative AI
}

# --- Retriever Initialization ---

# Sparse Retriever: BM25
# Tokenize documents for BM25
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

# Dense Retriever: Sentence Transformer
# Using a small, efficient model for demonstration
# In 2026, you might use larger, more performant models or specialized fine-tuned models.
model_name = 'all-MiniLM-L6-v2'
dense_model = SentenceTransformer(model_name)
document_embeddings = dense_model.encode(documents, convert_to_tensor=True)

print(f"Initialized BM25 with {len(documents)} documents.")
print(f"Initialized SentenceTransformer '{model_name}' and encoded {len(documents)} documents.")

# --- Helper Functions ---

def reciprocal_rank_fusion(rank_lists, k=60):
    """
    Performs Reciprocal Rank Fusion (RRF) on multiple ranked lists.
    Args:
        rank_lists (list of list of tuples): Each inner list is a ranked list
                                            of (document_index, score) from a retriever.
        k (int): A constant used in the RRF formula (1 / (k + rank)).
    Returns:
        list of tuples: A single ranked list of (document_index, fused_score).
    """
    fused_scores = collections.defaultdict(float)
    for rank_list in rank_lists:
        for rank, (doc_idx, _) in enumerate(rank_list):
            fused_scores[doc_idx] += 1.0 / (k + rank + 1) # +1 because rank is 0-indexed

    # Sort documents by their fused scores in descending order
    sorted_fused_scores = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)
    return sorted_fused_scores

def calculate_recall_at_k(retrieved_docs_indices, ground_truth_indices, k):
    """
    Calculates Recall@k for a single query.
    Args:
        retrieved_docs_indices (list): List of indices of documents retrieved by the system.
        ground_truth_indices (list): List of indices of truly relevant documents.
        k (int): The number of top documents to consider for recall.
    Returns:
        float: Recall@k score.
    """
    if not ground_truth_indices:
        return 1.0 # If no relevant documents, recall is 1 if nothing is missed.

    retrieved_at_k = set(retrieved_docs_indices[:k])
    relevant_at_k = set(ground_truth_indices)

    # Number of relevant documents found within the top k
    hits = len(retrieved_at_k.intersection(relevant_at_k))

    # Recall is hits divided by total number of relevant documents
    return hits / len(relevant_at_k)

print("Helper functions `reciprocal_rank_fusion` and `calculate_recall_at_k` defined.")


### Your Turn: Implement Retrieval Functions and Benchmark!

Now, it's your turn to implement the core retrieval logic. Complete the functions below and then run the benchmarking section to evaluate their performance. Remember to add comments to your code to explain your implementation choices.

#### Instructions:
1.  **`sparse_retrieve(query, top_n)`**: Implement BM25 retrieval.
2.  **`dense_retrieve(query, top_n)`**: Implement dense retrieval using cosine similarity.
3.  **`hybrid_retrieve(query, top_n, rrf_k)`**: Combine results from sparse and dense retrievers using `reciprocal_rank_fusion`.
4.  **Run Benchmarking**: Execute the provided benchmarking code block to see the results.


In [ ]:
# --- Student Implementation Area ---

def sparse_retrieve(query: str, top_n: int = 5) -> list[tuple[int, float]]:
    """
    Performs sparse retrieval using BM25.
    Args:
        query (str): The search query.
        top_n (int): The number of top documents to retrieve.
    Returns:
        list of tuples: A list of (document_index, score) for the top_n documents.
    """
    # Tokenize the query for BM25
    tokenized_query = query.lower().split()
    
    # Get BM25 scores for all documents
    doc_scores = bm25.get_scores(tokenized_query)
    
    # Pair document indices with their scores
    scored_docs = [(i, score) for i, score in enumerate(doc_scores)]
    
    # Sort by score in descending order and take top_n
    sorted_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    
    return sorted_docs[:top_n]

def dense_retrieve(query: str, top_n: int = 5) -> list[tuple[int, float]]:
    """
    Performs dense retrieval using SentenceTransformer embeddings and cosine similarity.
    Args:
        query (str): The search query.
        top_n (int): The number of top documents to retrieve.
    Returns:
        list of tuples: A list of (document_index, score) for the top_n documents.
    """
    # Encode the query into a dense vector
    query_embedding = dense_model.encode(query, convert_to_tensor=True)
    
    # Calculate cosine similarity between query and all document embeddings
    # Reshape query_embedding for sklearn's cosine_similarity if it's a single vector
    similarities = cosine_similarity(query_embedding.reshape(1, -1), document_embeddings).flatten()
    
    # Pair document indices with their similarity scores
    scored_docs = [(i, score) for i, score in enumerate(similarities)]
    
    # Sort by score in descending order and take top_n
    sorted_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    
    return sorted_docs[:top_n]

def hybrid_retrieve(query: str, top_n: int = 5, rrf_k: int = 60) -> list[tuple[int, float]]:
    """
    Performs hybrid retrieval by combining sparse and dense results using RRF.
    Args:
        query (str): The search query.
        top_n (int): The number of top documents to retrieve after fusion.
        rrf_k (int): The constant 'k' for Reciprocal Rank Fusion.
    Returns:
        list of tuples: A list of (document_index, fused_score) for the top_n documents.
    """
    # Get results from sparse retriever (e.g., top 100 for better RRF input)
    sparse_results = sparse_retrieve(query, top_n=100) # Retrieve more for RRF
    
    # Get results from dense retriever (e.g., top 100 for better RRF input)
    dense_results = dense_retrieve(query, top_n=100) # Retrieve more for RRF
    
    # Combine the ranked lists using RRF
    fused_results = reciprocal_rank_fusion([sparse_results, dense_results], k=rrf_k)
    
    # Return the top_n documents from the fused list
    return fused_results[:top_n]

# --- Benchmarking Section ---

def benchmark_retriever(retriever_func, queries, ground_truth, k_values=[1, 3, 5]):
    """
    Benchmarks a given retriever function across multiple queries and k values.
    Args:
        retriever_func (callable): The retrieval function to benchmark.
        queries (list): List of query strings.
        ground_truth (dict): Mapping of query index to relevant document indices.
        k_values (list): List of k values for Recall@k calculation.
    Returns:
        dict: A dictionary containing average Recall@k for each k.
    """
    results = {k: [] for k in k_values}
    
    for i, query in enumerate(queries):
        # Retrieve a sufficient number of documents to cover all k_values
        # We retrieve up to max(k_values) documents, or more if the retriever_func has its own top_n
        retrieved_docs = retriever_func(query, top_n=max(k_values) if 'rrf_k' not in retriever_func.__code__.co_varnames else 100)
        retrieved_indices = [doc_idx for doc_idx, _ in retrieved_docs]
        
        for k in k_values:
            recall = calculate_recall_at_k(retrieved_indices, ground_truth.get(i, []), k)
            results[k].append(recall)
            
    avg_results = {k: np.mean(recalls) for k, recalls in results.items()}
    return avg_results

print("\n--- Benchmarking Retrieval Methods ---")

k_values_to_evaluate = [1, 3, 5]

# Benchmark Sparse Retrieval
sparse_avg_recall = benchmark_retriever(sparse_retrieve, queries, ground_truth, k_values_to_evaluate)
print(f"Sparse Retrieval (BM25) Average Recall: {sparse_avg_recall}")

# Benchmark Dense Retrieval
dense_avg_recall = benchmark_retriever(dense_retrieve, queries, ground_truth, k_values_to_evaluate)
print(f"Dense Retrieval (SentenceTransformer) Average Recall: {dense_avg_recall}")

# Benchmark Hybrid Retrieval (RRF)
# Note: rrf_k parameter for hybrid_retrieve is passed implicitly by benchmark_retriever
# or can be set as a default in hybrid_retrieve itself.
# For benchmarking, we'll assume the default rrf_k=60 is used.
hybrid_avg_recall = benchmark_retriever(hybrid_retrieve, queries, ground_truth, k_values_to_evaluate)
print(f"Hybrid Retrieval (RRF) Average Recall: {hybrid_avg_recall}")

print("\n--- Analysis ---")
print("Observe how Hybrid Retrieval often outperforms individual sparse or dense methods, especially for Recall@k values.")
print("Sparse retrieval excels with exact keyword matches, while dense retrieval handles semantic similarity better.")
print("RRF effectively combines their strengths by giving higher weight to documents that rank well in both lists.")
print("For example, a document ranked #1 by sparse and #10 by dense will likely get a higher fused score than one ranked #1 by sparse and #50 by dense, or vice-versa.")
